In [1]:
import json

def calc_crc16(data: bytes) -> int:
    """Расчет CRC-16 (Modbus)"""
    crc = 0xFFFF
    for byte in data:
        crc ^= byte
        for _ in range(8):
            if (crc & 0x0001) != 0:
                crc >>= 1
                crc ^= 0xA001
            else:
                crc >>= 1
    return crc & 0xFFFF

def load_sensor_config(config_file: str = "sensor_config.json") -> dict:
    """Загрузка конфигурации сенсоров из JSON файла"""
    try:
        with open(config_file, 'r', encoding='utf-8') as f:
            return json.load(f)
    except FileNotFoundError:
        print(f"Warning: Config file {config_file} not found.")
        return {"field_mapping": {}}
def decode_sensor_data(data_bytes: bytes, id_sens: int, config: dict) -> dict:
    """
    Расшифровка данных сенсора на основе конфигурации
    Каждый параметр занимает 2 байта (16 бит)
    byte в конфиге - это номер 16-битного слова (начиная с 1)
    """
    result = {
        'id_sens': id_sens,
        'raw_data': data_bytes.hex().upper(),
        'decoded': {}
    }
    
    sensor_id = str(id_sens)
    field_map = config.get('field_mapping', {}).get(sensor_id, [])
    
    if not field_map:
        result['decoded'] = {'raw': data_bytes.hex(), 'note': 'No config for this sensor'}
        return result
    
    # Декодируем каждые 2 байта как 16-битное значение
    decoded_values = {}
    for field in field_map:
        word_num = field['byte']  # номер 16-битного слова (начиная с 1)
        field_name = field['name']
        coefficient = field['coef']
        
        # Вычисляем позицию байтов (каждое слово = 2 байта)
        byte_offset = (word_num - 1) * 2
        
        # Проверяем, есть ли эти байты в данных
        if byte_offset + 2 <= len(data_bytes):
            # Читаем 2 байта как 16-битное число (Big Endian)
            raw_value = int.from_bytes(data_bytes[byte_offset:byte_offset+2], 'big')
            calculated_value = raw_value * coefficient
            
            decoded_values[field_name] = {
                'word_position': word_num,
                'byte_offset': byte_offset,
                'raw_bytes': data_bytes[byte_offset:byte_offset+2].hex().upper(),
                'raw_value': raw_value,
                'coefficient': coefficient,
                'value': calculated_value
            }
        else:
            decoded_values[field_name] = {
                'word_position': word_num,
                'error': 'Not enough data'
            }
    
    result['decoded'] = decoded_values
    return result

def parse_packet(hex_string: str, config: dict = None) -> dict:
    """Парсинг пакета с расшифровкой данных сенсоров"""
    if config is None:
        config = load_sensor_config()
    
    data = bytes.fromhex(hex_string.strip())
    result = {}
    offset = 0
    
    # === ЗАГОЛОВОК ===
    result['ccid'] = data[0:10].hex().upper()
    offset = 10
    
    result['module_id'] = int.from_bytes(data[offset:offset+3], 'big')
    offset += 3
    
    result['bat_tx'] = data[offset] / 10.0
    offset += 1
    
    result['signal'] = data[offset]
    offset += 1
    
    result['rssi'] = data[offset]
    offset += 1
    
    # === ПОДВАЛ (от LEN) ===
    len_offset = offset
    packet_len = data[offset]
    result['packet_len'] = packet_len
    offset += 1
    
    result['sensor_module_id'] = int.from_bytes(data[offset:offset+3], 'big')
    offset += 3
    
    result['bat_mem'] = data[offset] / 10.0
    offset += 1
    
    date_bytes = data[offset:offset+6]
    result['date'] = f"20{date_bytes[0]:02d}-{date_bytes[1]:02d}-{date_bytes[2]:02d} {date_bytes[3]:02d}:{date_bytes[4]:02d}:{date_bytes[5]:02d}"
    offset += 6
    
    # === БЛОКИ СЕНСОРОВ ===
    sensors = []
    while offset < len_offset + packet_len - 2:
        if offset >= len(data):
            break
            
        port = data[offset]
        offset += 1
        
        if port == 0:
            break
        
        sens_start = offset
        id_sens = data[offset]
        operation = data[offset+1]
        len_pas = data[offset+2]
        offset += 3
        
        sensor_data = data[offset:offset+len_pas]
        offset += len_pas
        
        sensor_crc = int.from_bytes(data[offset:offset+2], 'little')
        offset += 2
        
        crc_data = data[sens_start:offset-2]
        calc_crc = calc_crc16(crc_data)
        
        # Расшифровка данных сенсора
        decoded_data = decode_sensor_data(sensor_data, id_sens, config)
        
        sensors.append({
            'port': port,
            'id_sens': id_sens,
            'operation': operation,
            'len_pas': len_pas,
            'data': sensor_data.hex().upper(),
            'decoded': decoded_data,
            'crc_sensor': sensor_crc,
            'crc_valid': sensor_crc == calc_crc
        })
    
    result['sensors'] = sensors
    
    # === ОБЩИЙ CRC ===
    total_data = data[len_offset:offset]
    total_crc = int.from_bytes(data[offset:offset+2], 'little')
    calc_total_crc = calc_crc16(total_data)
    
    result['crc_total'] = total_crc
    result['crc_total_valid'] = total_crc == calc_total_crc
    
    return result
def print_parsed_packet(parsed: dict):
    """Красивый вывод расшифрованного пакета"""
    print("=" * 80)
    print("                         РАСШИФРОВКА ПАКЕТА")
    print("=" * 80)
    
    print(f"\n📡 ЗАГОЛОВОК:")
    print(f"   CCID: {parsed['ccid']}")
    print(f"   Module ID: {parsed['module_id']}")
    print(f"   Battery (TX): {parsed['bat_tx']}V")
    print(f"   Signal: {parsed['signal']}% | RSSI: {parsed['rssi']}%")
    
    print(f"\n💾 ДАННЫЕ ПАМЯТИ:")
    print(f"   Packet Length: {parsed['packet_len']} bytes")
    print(f"   Sensor Module ID: {parsed['sensor_module_id']}")
    print(f"   Battery (MEM): {parsed['bat_mem']}V")
    print(f"   Date: {parsed['date']}")
    
    print(f"\n📊 СЕНСОРЫ ({len(parsed['sensors'])} шт.):")
    print("-" * 80)
    
    for i, sensor in enumerate(parsed['sensors'], 1):
        print(f"\n🔌 Сенсор #{i} (Port {sensor['port']}):")
        print(f"   ID Sensor (адрес): {sensor['id_sens']}")
        print(f"   Operation: 0x{sensor['operation']:02X}")
        print(f"   Data Length: {sensor['len_pas']} bytes")
        print(f"   Raw Data: {sensor['data']}")
        print(f"   CRC: 0x{sensor['crc_sensor']:04X} {'✓' if sensor['crc_valid'] else '✗'}")
        
        # Вывод расшифрованных данных
        if sensor['decoded'].get('decoded') and 'raw' not in sensor['decoded']:
            print(f"   📈 Расшифрованные параметры:")
            for param_name, param_data in sensor['decoded']['decoded'].items():
                if 'error' not in param_data:
                    value = param_data['value']
                    raw = param_data['raw_value']
                    coef = param_data['coefficient']
                    word_pos = param_data['word_position']
                    raw_bytes = param_data['raw_bytes']
                    print(f"      [{word_pos}] {param_name:25s}: {value:>10} (raw: {raw:>5}, {raw_bytes}, ×{coef})")
                else:
                    print(f"      {param_name}: {param_data['error']}")
    
    print("\n" + "=" * 80)
    print(f"✅ Общий CRC: 0x{parsed['crc_total']:04X} {'VALID' if parsed['crc_total_valid'] else 'INVALID'}")
    print("=" * 80)

# Пример использования
if __name__ == "__main__":
    # Загрузка конфигурации
    config = load_sensor_config("sensor_config.json")
    
    # Ваш пакет (сенсор 8 с 4 байтами данных)
    hex_str = "89701018292556469120000015224D0F2B000015281A0501041A2701080304000000FAE3700407030E000001040000001E0000000000001A460037"
    
    # Парсинг
    parsed = parse_packet(hex_str, config)
    
    # Вывод
    print_parsed_packet(parsed)
    
    # Сохранение результата в JSON
    with open('parsed_packet.json', 'w', encoding='utf-8') as f:
        json.dump(parsed, f, indent=2, ensure_ascii=False)
    print("\n💾 Результат сохранен в parsed_packet.json")

                         РАСШИФРОВКА ПАКЕТА

📡 ЗАГОЛОВОК:
   CCID: 89701018292556469120
   Module ID: 21
   Battery (TX): 3.4V
   Signal: 77% | RSSI: 15%

💾 ДАННЫЕ ПАМЯТИ:
   Packet Length: 43 bytes
   Sensor Module ID: 21
   Battery (MEM): 4.0V
   Date: 2026-05-01 04:26:39

📊 СЕНСОРЫ (2 шт.):
--------------------------------------------------------------------------------

🔌 Сенсор #1 (Port 1):
   ID Sensor (адрес): 8
   Operation: 0x03
   Data Length: 4 bytes
   Raw Data: 000000FA
   CRC: 0x70E3 ✓
   📈 Расшифрованные параметры:
      [1] leaf_humidity            :        0.0 (raw:     0, 0000, ×0.1)
      [2] leaf_temperature         :       25.0 (raw:   250, 00FA, ×0.1)

🔌 Сенсор #2 (Port 4):
   ID Sensor (адрес): 7
   Operation: 0x03
   Data Length: 14 bytes
   Raw Data: 000001040000001E000000000000
   CRC: 0x461A ✓
   📈 Расшифрованные параметры:
      [1] soil_humidity            :        0.0 (raw:     0, 0000, ×0.1)
      [2] soil_temperature         :       26.0 (raw:   260, 010

In [3]:
import json

def calc_crc16(data: bytes) -> int:
    crc = 0xFFFF
    for byte in data:
        crc ^= byte
        for _ in range(8):
            crc = (crc >> 1) ^ 0xA001 if (crc & 1) else (crc >> 1)
    return crc & 0xFFFF

def load_config(path="sensor_config.json") -> dict:
    try:
        with open(path, 'r', encoding='utf-8') as f: return json.load(f)
    except FileNotFoundError: return {"field_mapping": {}}

def parse_wh65lp(raw: bytes) -> dict:
    if len(raw) < 21: return {"error": "Too short"}
    d = raw
    # Wind Dir (9 bit)
    dir_val = d[2] + ((d[3] & 0x80) << 1)
    # Temp (11 bit)
    tmp_val = d[4] + ((d[3] & 0x07) << 8)
    # Humidity
    hum = d[5]
    # Wind Speed (9/10 bit)
    wsp = d[6] + ((d[3] & 0x10) << 4) if (d[3] & 0x04) else d[6]
    # Gust
    gust = d[7]
    # Rain (16 bit)
    rain = (d[8] << 8) | d[9]
    # UV (16 bit)
    uv = (d[10] << 8) | d[11]
    # Light (24 bit)
    light = (d[12] << 16) | (d[13] << 8) | d[14]
    # Pressure (24 bit)
    pres = (d[17] << 16) | (d[18] << 8) | d[19] if len(d) > 19 else 0
    
    return {
        "wind_direction": dir_val if dir_val != 0x1FF else 0,
        "temperature": (tmp_val - 400) / 10.0 if tmp_val != 0x7FF else 0,
        "humidity": hum if hum != 0xFF else 0,
        "wind_speed": (wsp / 8.0 * 0.51) if wsp != 0x1FF else 0,
        "wind_gust": gust * 0.51 if gust != 0xFF else 0,
        "rainfall": rain * 0.254,
        "uv": uv if uv != 0xFFFF else 0,
        "light": light / 10.0 if light != 0xFFFFFF else 0,
        "pressure": pres / 100.0 if pres != 0xFFFFFF else 0
    }

def wh65lp_to_meteo(weather: dict) -> bytes:
    arr = [
        weather["temperature"] * 10, weather["humidity"], weather["pressure"] * 10,
        weather["wind_direction"], weather["wind_speed"], weather["wind_gust"],
        weather["uv"], weather["light"] / 5, weather["rainfall"] * 100
    ]
    out = bytearray(18)
    for i in range(9):
        v = int(arr[i]) & 0xFFFF
        out[i*2], out[i*2+1] = (v >> 8) & 0xFF, v & 0xFF
    return bytes(out)

def decode_sensor(data: bytes, sid: int, cfg: dict) -> dict:
    res = {"id_sens": sid, "raw": data.hex().upper(), "decoded": {}}
    fmap = cfg.get("field_mapping", {}).get(str(sid), [])
    if not fmap:
        res["decoded"] = {"raw": data.hex(), "note": "Нет конфигурации"}
        return res
    for f in fmap:
        off = (f["byte"] - 1) * 2
        if off + 2 <= len(data):
            raw = int.from_bytes(data[off:off+2], "big")
            res["decoded"][f["name"]] = {
                "raw": raw, "coef": f["coef"], "value": raw * f["coef"]
            }
    return res

def parse_packet(hex_str: str, config: dict = None) -> dict:
    if config is None: config = load_config()
    data = bytes.fromhex(hex_str.strip())
    res, off = {}, 0
    
    # Header
    res["header"] = {
        "ccid": data[0:10].hex().upper(),
        "module_id": int.from_bytes(data[10:13], "big"),
        "bat_tx": data[13]/10, "signal": data[14], "rssi": data[15]
    }
    off = 16
    
    # Footer
    len_off = off
    pkt_len = data[off]; off += 1
    res["footer"] = {
        "pkt_len": pkt_len,
        "mod_id": int.from_bytes(data[off:off+3], "big"),
        "bat_mem": data[off+3]/10,
        "date": f"20{data[off+4]:02d}-{data[off+5]:02d}-{data[off+6]:02d} {data[off+7]:02d}:{data[off+8]:02d}:{data[off+9]:02d}"
    }
    off += 10
    
    # Sensors
    sensors = []
    while off < len(data) - 2:
        port = data[off]; off += 1
        if port == 0: break
        
        sid, op, dlen = data[off], data[off+1], data[off+2]
        off += 3
        sdata = data[off:off+dlen]; off += dlen
        scrc = int.from_bytes(data[off:off+2], "little"); off += 2
        
        # CRC сенсора (с ID до конца данных)
        crc_block = data[off-dlen-3 : off-2]
        valid_scrc = calc_crc16(crc_block) == scrc
        
        # Спец. обработка WH65LP
        if sid == 36 and dlen == 21:
            weather = parse_wh65lp(sdata)
            meteo = wh65lp_to_meteo(weather)
            parsed = decode_sensor(meteo, sid, config)
            parsed["wh65lp_raw"] = weather
        else:
            parsed = decode_sensor(sdata, sid, config)
            
        sensors.append({
            "port": port, "id": sid, "op": op, "len": dlen,
            "data": sdata.hex().upper(), "crc": scrc, "crc_valid": valid_scrc,
            "parsed": parsed
        })
    res["sensors"] = sensors
    
    # Total CRC
    total_block = data[len_off:off]
    tcrc = int.from_bytes(data[off:off+2], "little")
    res["crc_total"] = tcrc
    res["crc_total_valid"] = calc_crc16(total_block) == tcrc
    return res

# === ТЕСТ ===
if __name__ == "__main__":
    hex_input="89 70 10 18 29 25 56 46 91 46 00 4E 21 29 40 00 2B 00 4E 21 00 1A 05 03 01 2A 06 01 08 03 04 00 00 00 F5 A3 74 04 07 03 0E 00 00 00 FC 00 00 00 1E 00 00 00 00 00 00 CE E6 5E 3A"
    result = parse_packet(hex_input)
    print(json.dumps(result, indent=2, ensure_ascii=False))

{
  "header": {
    "ccid": "89701018292556469146",
    "module_id": 20001,
    "bat_tx": 4.1,
    "signal": 64,
    "rssi": 0
  },
  "footer": {
    "pkt_len": 43,
    "mod_id": 20001,
    "bat_mem": 0.0,
    "date": "2026-05-03 01:42:06"
  },
  "sensors": [
    {
      "port": 1,
      "id": 8,
      "op": 3,
      "len": 4,
      "data": "000000F5",
      "crc": 29859,
      "crc_valid": false,
      "parsed": {
        "id_sens": 8,
        "raw": "000000F5",
        "decoded": {
          "leaf_humidity": {
            "raw": 0,
            "coef": 0.1,
            "value": 0.0
          },
          "leaf_temperature": {
            "raw": 245,
            "coef": 0.1,
            "value": 24.5
          }
        }
      }
    },
    {
      "port": 4,
      "id": 7,
      "op": 3,
      "len": 14,
      "data": "000000FC0000001E000000000000",
      "crc": 59086,
      "crc_valid": false,
      "parsed": {
        "id_sens": 7,
        "raw": "000000FC0000001E000000000000",
    